# Search-7-MCTS-And-Beyond : Monte Carlo Tree Search et Extensions

**Navigation** : [<< Recherche adversariale](Search-6-AdversarialSearch.ipynb) | [Index](../README.md) | [Dancing Links >>](Search-8-DancingLinks.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. **Comprendre** les limites de Minimax et pourquoi MCTS a revolutionne les jeux
2. **Implementer** l'algorithme MCTS avec UCB1
3. **Comparer** MCTS vs Minimax sur differents types de jeux
4. **Decouvrir** OpenSpiel, le framework de Google pour les jeux
5. **Explorer** les approches hybrides (AlphaGo, AlphaZero)

### Prerequis
- Notebook Search-6 (AdversarialSearch, Minimax, Alpha-Beta)
- Bases de probabilites et statistiques
- Bases de Python : classes, recursion

### Duree estimee : 90 minutes

In [1]:
# Imports
import sys
import time
import random
import math
from typing import Optional, List, Dict, Tuple, Any
from copy import deepcopy
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline

print("Environnement pret pour MCTS.")

Environnement pret pour MCTS.


## 1. Limites de Minimax

### Pourquoi Minimax ne suffit pas

L'algorithme Minimax avec Alpha-Beta fonctionne bien pour les jeux simples, mais rencontre des limites :

1. **Explosion combinatoire** : Aux echecs, meme avec Alpha-Beta, on ne peut explorer que 6-8 demi-coups en profondeur
2. **Fonction d'evaluation** : Necessite une expertise humaine pour concevoir une bonne heuristique
3. **Horizon effect** : Des evenements importants au-dela de la profondeur sont ignores

### La revolution AlphaGo (2016)

AlphaGo a battu Lee Sedol (champion du monde de Go) en utilisant :
- **MCTS** pour la recherche
- **Reseaux de neurones** pour l'evaluation (policy + value networks)

MCTS permet d'explorer intelligemment sans fonction d'evaluation experte !

## 2. L'Algorithme MCTS

### Principe

**Monte Carlo Tree Search** construit progressivement un arbre de recherche en :
1. **Selection** : Traverser l'arbre avec UCB1 pour equilibrer exploration/exploitation
2. **Expansion** : Ajouter un nouveau noeud a l'arbre
3. **Simulation** : Jouer une partie aleatoire jusqu'a la fin (rollout)
4. **Backpropagation** : Remonter le resultat dans l'arbre

### UCB1 (Upper Confidence Bound)

UCB1 equilibre exploration et exploitation :

```
UCB1 = W/N + c * sqrt(ln(N_parent) / N)
```

- **W/N** : Taux de victoire (exploitation)
- **c * sqrt(...)** : Terme d'exploration (c = 1.41 typiquement)
- **N** : Nombre de visites du noeud
- **N_parent** : Nombre de visites du parent

In [2]:
class NoeudMCTS:
    """Noeud de l'arbre MCTS."""
    
    def __init__(self, etat: Any, parent=None, action=None):
        self.etat = etat
        self.parent = parent
        self.action = action  # Action qui a mene a cet etat
        self.enfants = {}     # action -> NoeudMCTS
        self.visites = 0
        self.victoires = 0.0
        self.actions_non_explorees = None  # Initialise lors de la premiere expansion
    
    def ucb1(self, c: float = 1.41) -> float:
        """Calcul du score UCB1."""
        if self.visites == 0:
            return float('inf')
        
        exploitation = self.victoires / self.visites
        exploration = c * math.sqrt(math.log(self.parent.visites) / self.visites)
        return exploitation + exploration
    
    def meilleur_enfant_ucb1(self, c: float = 1.41) -> 'NoeudMCTS':
        """Selectionne l'enfant avec le meilleur score UCB1."""
        return max(self.enfants.values(), key=lambda n: n.ucb1(c))
    
    def meilleur_enfant_visites(self) -> 'NoeudMCTS':
        """Selectionne l'enfant le plus visite (pour le coup final)."""
        return max(self.enfants.values(), key=lambda n: n.visites)
    
    def est_fully_expanded(self, jeu) -> bool:
        """Verifie si toutes les actions ont ete expandues."""
        if self.actions_non_explorees is None:
            self.actions_non_explorees = list(jeu.actions(self.etat))
        return len(self.actions_non_explorees) == 0
    
    def est_terminal(self, jeu) -> bool:
        return jeu.est_terminal(self.etat)

Implementation de la classe MCTS avec selection UCB1, expansion, simulation et retropropagation.

In [3]:
class MCTS:
    """Implementation de Monte Carlo Tree Search."""
    
    def __init__(self, jeu, c: float = 1.41):
        self.jeu = jeu
        self.c = c  # Parametre d'exploration
        self.stats = {'selections': 0, 'expansions': 0, 'simulations': 0, 'backprops': 0}
    
    def recherche(self, etat: Any, iterations: int = 1000) -> Tuple[Any, float]:
        """
        Execute MCTS depuis l'etat donne.
        Retourne (meilleure_action, valeur_estimee).
        """
        racine = NoeudMCTS(etat)
        
        for _ in range(iterations):
            noeud = self._selection(racine)
            resultat = self._simulation(noeud)
            self._backpropagation(noeud, resultat)
        
        meilleur = racine.meilleur_enfant_visites()
        valeur = meilleur.victoires / meilleur.visites if meilleur.visites > 0 else 0
        return meilleur.action, valeur
    
    def _selection(self, noeud: NoeudMCTS) -> NoeudMCTS:
        """Descend dans l'arbre jusqu'a trouver un noeud a explorer."""
        self.stats['selections'] += 1
        
        while not noeud.est_terminal(self.jeu):
            if not noeud.est_fully_expanded(self.jeu):
                return self._expansion(noeud)
            else:
                noeud = noeud.meilleur_enfant_ucb1(self.c)
        
        return noeud
    
    def _expansion(self, noeud: NoeudMCTS) -> NoeudMCTS:
        """Ajoute un nouvel enfant a l'arbre."""
        self.stats['expansions'] += 1
        
        if noeud.actions_non_explorees is None:
            noeud.actions_non_explorees = list(self.jeu.actions(noeud.etat))
        
        action = noeud.actions_non_explorees.pop()
        nouvel_etat = self.jeu.resultat(noeud.etat, action)
        enfant = NoeudMCTS(nouvel_etat, parent=noeud, action=action)
        noeud.enfants[action] = enfant
        return enfant
    
    def _simulation(self, noeud: NoeudMCTS) -> float:
        """
        Simule une partie aleatoire depuis le noeud.
        Retourne le resultat du point de vue du joueur MAX.
        """
        self.stats['simulations'] += 1
        
        etat = noeud.etat
        joueur_max_original = self.jeu.joueur(noeud.parent.etat) if noeud.parent else 'MAX'
        
        while not self.jeu.est_terminal(etat):
            actions = list(self.jeu.actions(etat))
            action = random.choice(actions)
            etat = self.jeu.resultat(etat, action)
        
        return self.jeu.utilite(etat, joueur_max_original)
    
    def _backpropagation(self, noeud: NoeudMCTS, resultat: float):
        """Remonte le resultat dans l'arbre."""
        self.stats['backprops'] += 1
        
        while noeud is not None:
            noeud.visites += 1
            
            # Le resultat depend du point de vue du joueur a ce noeud
            if self.jeu.joueur(noeud.etat) == 'MAX':
                noeud.victoires += resultat
            else:
                noeud.victoires += -resultat  # Inverser pour MIN
            
            noeud = noeud.parent

Maintenant que nous avons la structure de nœud, nous pouvons implémenter l'algorithme **MCTS complet** qui utilise ces nœuds pour construire l'arbre de recherche.


## 3. Test sur Tic-Tac-Toe

Comparons MCTS avec Minimax sur le jeu de Morpion.

In [4]:
# Reutilisation de la classe TicTacToe du notebook precedent
from abc import ABC, abstractmethod

class JeuSommeNulle(ABC):
    @abstractmethod
    def etat_initial(self) -> Any: pass
    @abstractmethod
    def joueur(self, etat: Any) -> str: pass
    @abstractmethod
    def actions(self, etat: Any) -> List[Any]: pass
    @abstractmethod
    def resultat(self, etat: Any, action: Any) -> Any: pass
    @abstractmethod
    def est_terminal(self, etat: Any) -> bool: pass
    @abstractmethod
    def utilite(self, etat: Any, joueur: str) -> float: pass

class TicTacToe(JeuSommeNulle):
    def __init__(self):
        self._etat_initial = (tuple([' ']*9), 'X')
    
    def etat_initial(self):
        return self._etat_initial
    
    def joueur(self, etat):
        return 'MAX' if etat[1] == 'X' else 'MIN'
    
    def actions(self, etat):
        return [i for i in range(9) if etat[0][i] == ' ']
    
    def resultat(self, etat, action):
        grille = list(etat[0])
        joueur = etat[1]
        grille[action] = joueur
        prochain = 'O' if joueur == 'X' else 'X'
        return (tuple(grille), prochain)
    
    def est_terminal(self, etat):
        grille = etat[0]
        lignes = [[0,1,2],[3,4,5],[6,7,8],[0,3,6],[1,4,7],[2,5,8],[0,4,8],[2,4,6]]
        for l in lignes:
            if grille[l[0]] != ' ' and grille[l[0]] == grille[l[1]] == grille[l[2]]:
                return True
        return ' ' not in grille
    
    def utilite(self, etat, joueur):
        grille = etat[0]
        lignes = [[0,1,2],[3,4,5],[6,7,8],[0,3,6],[1,4,7],[2,5,8],[0,4,8],[2,4,6]]
        for l in lignes:
            if grille[l[0]] != ' ' and grille[l[0]] == grille[l[1]] == grille[l[2]]:
                gagnant = 'MAX' if grille[l[0]] == 'X' else 'MIN'
                return 1 if gagnant == joueur else -1
        return 0

# Test MCTS
jeu = TicTacToe()
mcts = MCTS(jeu)

start = time.time()
action, valeur = mcts.recherche(jeu.etat_initial(), iterations=1000)
temps = time.time() - start

print(f"MCTS (1000 iterations): action={action}, valeur={valeur:.3f}, temps={temps:.3f}s")
print(f"Stats: {mcts.stats}")

MCTS (1000 iterations): action=3, valeur=0.065, temps=0.024s
Stats: {'selections': 1000, 'expansions': 1000, 'simulations': 1000, 'backprops': 1000}


## 4. Comparaison MCTS vs Minimax

Comparons les deux approches sur differents criteres.

In [5]:
# Minimax pour comparaison
def minimax(jeu, etat, joueur_max='MAX'):
    if jeu.est_terminal(etat):
        return jeu.utilite(etat, joueur_max), None
    
    actions = jeu.actions(etat)
    if jeu.joueur(etat) == joueur_max:
        best_v, best_a = float('-inf'), None
        for a in actions:
            v, _ = minimax(jeu, jeu.resultat(etat, a), joueur_max)
            if v > best_v:
                best_v, best_a = v, a
        return best_v, best_a
    else:
        best_v, best_a = float('+inf'), None
        for a in actions:
            v, _ = minimax(jeu, jeu.resultat(etat, a), joueur_max)
            if v < best_v:
                best_v, best_a = v, a
        return best_v, best_a

# Benchmark
resultats = []

# Minimax
start = time.time()
v_mm, a_mm = minimax(jeu, jeu.etat_initial())
t_mm = time.time() - start
resultats.append(('Minimax', t_mm, v_mm, a_mm))

# MCTS avec differents nombres d'iterations
for n_iter in [100, 500, 1000, 5000]:
    mcts = MCTS(jeu)
    start = time.time()
    v, a = mcts.recherche(jeu.etat_initial(), iterations=n_iter)
    t = time.time() - start
    resultats.append((f'MCTS ({n_iter})', t, v, a))

# Affichage
df = pd.DataFrame(resultats, columns=['Algorithme', 'Temps (s)', 'Valeur', 'Action'])
display(df)

,Algorithme,Temps (s),Valeur,Action
0,Minimax,1.209055,0,0.000000
1,MCTS (100),0.002193,0,0.263158
2,MCTS (500),0.013009,5,0.104167
3,MCTS (1000),0.030354,8,0.113712
4,MCTS (5000),0.125474,0,0.035533


### Analyse des resultats

| Critere | Minimax | MCTS |
|---------|---------|------|
| **Garantie d'optimalite** | Oui (si arbre complet) | Non (probabiliste) |
| **Fonction d'evaluation** | Necessaire | Non necessaire |
| **Complexite** | O(b^d) | O(n * d) ou n = iterations |
| **Parallellisable** | Difficile | Tres facile |
| **Jeux a grand facteur de branchement** | Impraticable | Adapté |

### Quand utiliser MCTS ?

- Jeux avec grand espace d'etat (Go, Hex)
- Jeux sans bonne fonction d'evaluation
- Situations avec contrainte de temps variable
- Jeux avec hasard (backgammon, poker)

## 5. OpenSpiel - Framework de Jeux

### Presentation

**OpenSpiel** est un framework open-source de DeepMind pour la recherche en IA sur les jeux.

### Caracteristiques

- **40+ jeux** : Echecs, Go, Hex, Poker, Hanabi, etc.
- **Multi-agent** : Jeux a N joueurs
- **Information imparfaite** : Poker, Hanabi
- **Algos inclus** : MCTS, AlphaZero, CFR, etc.

### Installation

```bash
# Linux/WSL uniquement
pip install open-spiel
```

In [6]:
# Note: OpenSpiel necessite Linux/WSL
# Ce code est un exemple de l'API (ne fonctionnera pas sur Windows natif)

openspiel_available = False

try:
    import pyspiel
    openspiel_available = True
    
    # Charger le jeu de Tic-Tac-Toe
    game = pyspiel.load_game("tic_tac_toe")
    state = game.new_initial_state()
    
    print(f"Jeu: {game}")
    print(f"Etat initial: {state}")
    
    # MCTS d'OpenSpiel
    from open_spiel.python.algorithms import mcts
    
    # Configuration MCTS
    rng = np.random.RandomState(42)
    evaluator = mcts.RandomRolloutEvaluator(1, rng)
    bot = mcts.MCTSBot(game, 2.0, 1000, evaluator, rng)
    
    # Jouer un coup
    action = bot.step(state)
    print(f"Action MCTS: {action}")
    
except ImportError:
    print("OpenSpiel n'est pas installe ou non disponible sur cette plateforme.")
    print("Installation: pip install open-spiel (Linux/WSL uniquement)")

OpenSpiel n'est pas installe ou non disponible sur cette plateforme.
Installation: pip install open-spiel (Linux/WSL uniquement)


## 6. AlphaGo et AlphaZero

### Architecture AlphaGo (2016)

AlphaGo combine MCTS avec des reseaux de neurones :

1. **Policy Network** : Predit la probabilite de chaque coup
2. **Value Network** : Evalue la position
3. **MCTS** : Guide la recherche avec les predictions des reseaux

### AlphaZero (2017)

AlphaZero simplifie et generalise l'approche :

- **Auto-apprentissage** : Pas de donnees humaines
- **Reseau unique** : Policy + Value ensemble
- **Universel** : Fonctionne pour Go, Echecs, Shogi

### Principe de l'apprentissage

```
1. Initialiser le reseau aleatoirement
2. Jouer des parties contre soi-meme avec MCTS
3. Entrainer le reseau sur les positions et resultats
4. Repeter jusqu'a convergence
```

In [7]:
# Sketch conceptuel d'un AlphaZero simplifie

class AlphaZeroMCTS:
    """
    MCTS guide par un reseau de neurones (version simplifiee).
    En pratique, utiliser des frameworks comme PyTorch ou TensorFlow.
    """
    
    def __init__(self, jeu, policy_value_fn, c=1.41):
        self.jeu = jeu
        self.policy_value_fn = policy_value_fn
        self.c = c
    
    def recherche(self, etat, iterations=100):
        """
        MCTS avec evaluation par reseau de neurones.
        policy_value_fn(etat) -> (probs_dict, value)
        """
        racine = NoeudMCTS(etat)
        
        for _ in range(iterations):
            noeud = racine
            chemin = [noeud]
            
            # Selection
            while noeud.enfants and noeud.est_fully_expanded(self.jeu):
                noeud = noeud.meilleur_enfant_ucb1(self.c)
                chemin.append(noeud)
            
            # Evaluation par le reseau
            if noeud.est_terminal(self.jeu):
                value = self.jeu.utilite(noeud.etat, 'MAX')
            else:
                probs, value = self.policy_value_fn(noeud.etat)
                
                # Expansion avec les probabilites du policy network
                for action in self.jeu.actions(noeud.etat):
                    enfant = NoeudMCTS(
                        self.jeu.resultat(noeud.etat, action),
                        parent=noeud,
                        action=action
                    )
                    noeud.enfants[action] = enfant
            
            # Backpropagation
            for n in chemin:
                n.visites += 1
                n.victoires += value
        
        return racine.meilleur_enfant_visites().action

def random_policy_value(etat):
    """Policy/Value network fictif (aleatoire) pour demonstration."""
    jeu = TicTacToe()
    actions = list(jeu.actions(etat))
    probs = {a: 1.0/len(actions) for a in actions}
    value = random.uniform(-0.5, 0.5)
    return probs, value

print("Sketch AlphaZero implemente (conceptuel).")

Sketch AlphaZero implemente (conceptuel).


## 7. Extensions Avancees

### RAVE (Rapid Action Value Estimation)

RAVE utilise l'heuristique "all moves as first" pour accelerer l'apprentissage :
- Un coup est evalue meme s'il est joue plus tard dans la partie
- Acceleration significative dans les premiers stades

### UCT (UCB applied to Trees)

UCT est le nom original de l'algorithme MCTS avec UCB1. Variantes :
- **UCB1-Tuned** : Ajuste le parametre d'exploration
- **UCB-V** : Utilise la variance des recompenses

### Parallelisation

MCTS se parallellise facilement :
- **Leaf parallelisation** : Simulations en parallele
- **Root parallelisation** : Plusieurs arbres en parallele
- **Tree parallelisation** : Acces concurrent a l'arbre

In [8]:
# Exemple de parallelisation simple avec multiprocessing
from multiprocessing import Pool

def mcts_worker(args):
    """Worker pour parallelisation."""
    jeu, etat, iterations = args
    mcts = MCTS(jeu)
    action, _ = mcts.recherche(etat, iterations)
    return action

def mcts_parallel(jeu, etat, total_iterations=1000, n_workers=4):
    """
    MCTS parallele avec root parallelisation.
    """
    iterations_per_worker = total_iterations // n_workers
    args = [(jeu, etat, iterations_per_worker) for _ in range(n_workers)]
    
    with Pool(n_workers) as pool:
        actions = pool.map(mcts_worker, args)
    
    # Vote majoritaire
    from collections import Counter
    vote = Counter(actions)
    return vote.most_common(1)[0][0]

print("MCTS parallele implemente (ne fonctionne pas dans un notebook interactif).")

MCTS parallele implemente (ne fonctionne pas dans un notebook interactif).


## 8. Synthese

### Resume des approches

| Approche | Force | Faiblesse |
|----------|-------|-----------|
| **Minimax + Alpha-Beta** | Optimal si arbre complet | Explosion combinatoire |
| **MCTS pur** | Pas de fonction d'evaluation | Convergence lente |
| **MCTS + Heuristiques** | Rapide, bon compromis | Heuristiques necessaires |
| **AlphaZero** | Auto-apprentissage | Ressources enormes |

### Quand utiliser quoi ?

- **Jeux simples** (Tic-Tac-Toe, petits puzzles) : Minimax
- **Jeux moyens** (Connect Four, Othello) : Alpha-Beta + heuristiques
- **Jeux complexes** (Go, Hex) : MCTS + reseaux de neurones
- **Jeux a information imparfaite** (Poker, Hanabi) : CFR + deep learning

### Pour aller plus loin

- **MuZero** : Apprend les regles du jeu (model-based)
- **AlphaFold** : Applications hors jeux (biologie)
- **Expert Iteration** : Combinaison expert/apprenti

---

**Navigation** : [<< Recherche adversariale](Search-6-AdversarialSearch.ipynb) | [Index](../README.md) | [Dancing Links >>](Search-8-DancingLinks.ipynb)

## Exercices

### Exercice 1 : Parametre d'exploration c
Testez differentes valeurs du parametre c (0.5, 1.0, 1.41, 2.0) et analysez l'impact sur les performances.

### Exercice 2 : Rollout intelligent
Implementez un rollout guide par heuristiques au lieu d'un choix purement aleatoire.

### Exercice 3 : MCTS vs Alpha-Beta sur Connect Four
Comparez MCTS et Alpha-Beta sur le jeu de Puissance 4.

### Exercice 4 : Extension RAVE
Implementez l'extension RAVE (Rapid Action Value Estimation) pour accelerer la convergence.

### Exercice 5 : Time management
Implementez une gestion du temps adaptive pour MCTS.

### Exercice 6 : MCTS sur le jeu de Nim
Implementez le jeu de Nim (prendre 1 a 3 allumettes, le dernier perd) et verifiez que MCTS decouvre la strategie optimale connue (laisser un multiple de 4).

### Exercice 7 : Analyse de convergence
Mesurez comment la qualite des decisions MCTS evolue avec le nombre d'iterations et tracez les courbes de convergence.

In [ ]:
# Exercice 7 : Analyse de convergence MCTS
# TODO: Mesurez comment la qualite de MCTS evolue avec le nombre d'iterations
# - Lancer MCTS avec des nombres d'iterations croissants (10, 50, 100, 500, 1000, 5000)
# - Pour chaque: repeter N fois et mesurer le taux de choix de l'action optimale
# - Comparer l'action MCTS avec l'action Minimax (verite terrain)
# - Tracer: iterations vs taux d'action optimale, iterations vs valeur estimee
# Indice: l'action Minimax sur TicTacToe est la reference

def analyser_convergence(jeu, etat, iterations_list, repetitions=20):
    """
    Analyse la convergence de MCTS.
    Pour chaque nombre d'iterations, repete 'repetitions' fois et collecte:
    - Frequence de l'action optimale (comparee a Minimax)
    - Valeur estimee moyenne
    - Temps moyen
    """
    # TODO: Implementer l'analyse
    # 1. Trouver l'action optimale avec Minimax (reference)
    # 2. Pour chaque n dans iterations_list:
    #    - Lancer MCTS 'repetitions' fois
    #    - Compter combien de fois MCTS choisit l'action Minimax
    #    - Collecter valeur estimee et temps
    # 3. Retourner un DataFrame avec les resultats
    pass

def tracer_convergence(resultats_df):
    """
    Trace les courbes de convergence.
    Graphiques attendus:
    - Iterations vs taux d'action optimale (%)
    - Iterations vs valeur estimee moyenne
    - Iterations vs temps moyen
    """
    # TODO: Utiliser matplotlib pour creer 3 sous-graphiques
    # Utiliser une echelle log pour l'axe x (iterations)
    pass

raise NotImplementedError("A vous de jouer !")

---

In [ ]:
# Exercice 6 : MCTS sur le jeu de Nim
# TODO: Implementez le jeu de Nim et testez MCTS dessus
# - Regles: N allumettes au depart, chaque joueur prend 1, 2 ou 3 allumettes
# - Le joueur qui prend la derniere allumette PERD
# - Strategie optimale connue: laisser un multiple de 4 a l'adversaire
# - Comparez MCTS vs la strategie optimale sur plusieurs parties
# Indice: la classe doit heriter de JeuSommeNulle

class NimGame(JeuSommeNulle):
    """Jeu de Nim : N allumettes, prendre 1-3, le dernier a jouer perd."""

    def __init__(self, n_allumettes=15):
        # TODO: Stocker le nombre initial d'allumettes
        pass

    def etat_initial(self):
        # TODO: Retourner (n_allumettes_restantes, joueur_courant)
        pass

    def joueur(self, etat):
        # TODO: Retourner 'MAX' ou 'MIN'
        pass

    def actions(self, etat):
        # TODO: Retourner les prises possibles [1, 2, 3] (max = allumettes restantes)
        pass

    def resultat(self, etat, action):
        # TODO: Retirer 'action' allumettes et changer de joueur
        pass

    def est_terminal(self, etat):
        # TODO: Terminal quand 0 allumettes restantes
        pass

    def utilite(self, etat, joueur):
        # TODO: Le joueur qui a pris la derniere allumette perd
        # Le joueur courant dans un etat terminal est celui qui doit jouer
        # mais il n'y a plus d'allumettes = l'autre a pris la derniere = l'autre perd
        pass

def strategie_optimale_nim(etat):
    """
    Strategie optimale : laisser un multiple de 4 a l'adversaire.
    Retourne le nombre d'allumettes a prendre.
    """
    # TODO: Calculer n % 4, prendre ce reste (ou 1 si deja multiple de 4)
    pass

raise NotImplementedError("A vous de jouer !")

---

In [ ]:
# Exercice 5 : Time management
# TODO: Implementez une gestion du temps adaptive pour MCTS
# - Allouer plus de temps aux positions critiques
# - Utiliser moins de temps quand le coup est evident
# - Detecter les positions "faciles" (victoire probable)
# Indice: utiliser la variance des evaluations pour detecter l'incertitude

class MCTSTimeManaged(MCTS):
    """MCTS avec gestion adaptive du temps."""
    
    def recherche_adaptive(self, etat, temps_max=1.0, min_iterations=100):
        """
        MCTS avec gestion adaptive du temps.
        Arrete quand: temps ecoule OU confidence suffisante.
        """
        # TODO: Implementer la logique adaptive
        # 1. Si variance des evaluations faible, arreter plus tot
        # 2. Si victoire probable (valeur > 0.9), arreter
        # 3. Sinon, utiliser tout le temps
        pass
    
    def calculer_confidence(self, racine):
        """Calcule la confiance dans le meilleur coup."""
        # TODO: Utiliser l'ecart-type des valeurs des enfants
        pass

raise NotImplementedError("A vous de jouer !")

---

In [ ]:
# Exercice 4 : Extension RAVE
# TODO: Implementez RAVE (Rapid Action Value Estimation)
# - RAVE utilise l'heuristique "all moves as first"
# - Un coup est evalue meme s'il est joue plus tard
# - Formule: Q_RAVE = (1-beta) * Q_MCTS + beta * Q_RAVE
# Indice: modifier la classe NoeudMCTS pour stocker les stats RAVE

class NoeudRAVE(NoeudMCTS):
    """Noeud MCTS avec support RAVE."""
    
    def __init__(self, etat, parent=None, action=None):
        super().__init__(etat, parent, action)
        # TODO: Ajouter les attributs pour RAVE
        # rave_visites, rave_victoires
        pass
    
    def valeur_rave(self, beta):
        """Calcule la valeur combinee MCTS + RAVE."""
        # TODO: Implementer la formule RAVE
        pass

class MCTSRAVE(MCTS):
    """MCTS avec extension RAVE."""
    
    def _backpropagation(self, noeud, resultat, chemin_actions):
        """Backpropagation avec mise a jour RAVE."""
        # TODO: Mettre a jour les stats RAVE pour toutes les actions du chemin
        pass

raise NotImplementedError("A vous de jouer !")

---

In [ ]:
# Exercice 3 : MCTS vs Alpha-Beta sur Connect Four
# Implementation de Connect Four + tournoi entre les deux algorithmes

class ConnectFour(JeuSommeNulle):
    """
    Jeu de Puissance 4 (Connect Four).
    Grille 6 lignes x 7 colonnes.
    'X' = MAX (Rouge), 'O' = MIN (Jaune).
    """
    
    def __init__(self, lignes=6, colonnes=7):
        self.lignes = lignes
        self.colonnes = colonnes
    
    def etat_initial(self):
        """Etat = (grille 2D, joueur courant). Grille = liste de listes."""
        grille = tuple(tuple([' '] * self.colonnes) for _ in range(self.lignes))
        return (grille, 'X')
    
    def joueur(self, etat):
        return 'MAX' if etat[1] == 'X' else 'MIN'
    
    def actions(self, etat):
        """Colonnes ou on peut encore jouer (pas pleines)."""
        grille = etat[0]
        return [c for c in range(self.colonnes) if grille[0][c] == ' ']
    
    def resultat(self, etat, action):
        """Place le jeton dans la colonne (tombe vers le bas)."""
        grille = [list(ligne) for ligne in etat[0]]
        joueur = etat[1]
        
        # Trouver la ligne la plus basse disponible
        for ligne in range(self.lignes - 1, -1, -1):
            if grille[ligne][action] == ' ':
                grille[ligne][action] = joueur
                break
        
        prochain = 'O' if joueur == 'X' else 'X'
        return (tuple(tuple(l) for l in grille), prochain)
    
    def est_terminal(self, etat):
        grille = etat[0]
        # Verifier alignements horizontaux, verticaux, diagonaux
        for r in range(self.lignes):
            for c in range(self.colonnes):
                if grille[r][c] == ' ':
                    continue
                pion = grille[r][c]
                # Horizontal
                if c + 3 < self.colonnes and all(grille[r][c+i] == pion for i in range(4)):
                    return True
                # Vertical
                if r + 3 < self.lignes and all(grille[r+i][c] == pion for i in range(4)):
                    return True
                # Diagonal bas-droite
                if r + 3 < self.lignes and c + 3 < self.colonnes and all(grille[r+i][c+i] == pion for i in range(4)):
                    return True
                # Diagonal haut-droite
                if r - 3 >= 0 and c + 3 < self.colonnes and all(grille[r-i][c+i] == pion for i in range(4)):
                    return True
        
        # Match nul (grille pleine)
        return all(grille[0][c] != ' ' for c in range(self.colonnes))
    
    def utilite(self, etat, joueur):
        grille = etat[0]
        # Chercher le gagnant
        for r in range(self.lignes):
            for c in range(self.colonnes):
                if grille[r][c] == ' ':
                    continue
                pion = grille[r][c]
                gagnant = 'MAX' if pion == 'X' else 'MIN'
                # Verifier les 4 directions
                for dr, dc in [(0,1),(1,0),(1,1),(-1,1)]:
                    if 0 <= r+3*dr < self.lignes and 0 <= c+3*dc < self.colonnes:
                        if all(grille[r+i*dr][c+i*dc] == pion for i in range(4)):
                            return 1 if gagnant == joueur else -1
        return 0  # Match nul


# Alpha-Beta avec profondeur limitee
def alphabeta(jeu, etat, profondeur, alpha=float('-inf'), beta=float('+inf'), joueur_max='MAX'):
    """Alpha-Beta avec profondeur limitee."""
    if jeu.est_terminal(etat) or profondeur == 0:
        if jeu.est_terminal(etat):
            return jeu.utilite(etat, joueur_max), None
        return 0, None  # Heuristique neutre a profondeur max
    
    actions = jeu.actions(etat)
    if jeu.joueur(etat) == joueur_max:
        best_v, best_a = float('-inf'), None
        for a in actions:
            v, _ = alphabeta(jeu, jeu.resultat(etat, a), profondeur - 1, alpha, beta, joueur_max)
            if v > best_v:
                best_v, best_a = v, a
            alpha = max(alpha, v)
            if beta <= alpha:
                break
        return best_v, best_a
    else:
        best_v, best_a = float('+inf'), None
        for a in actions:
            v, _ = alphabeta(jeu, jeu.resultat(etat, a), profondeur - 1, alpha, beta, joueur_max)
            if v < best_v:
                best_v, best_a = v, a
            beta = min(beta, v)
            if beta <= alpha:
                break
        return best_v, best_a


def tournoi_mcts_vs_alphabeta(jeu, mcts_iterations=1000, alpha_beta_depth=6, parties=20):
    """
    Organise un tournoi entre MCTS et Alpha-Beta.
    Retourne les statistiques de victoire.
    """
    stats = {
        'mcts_victoires': 0,
        'ab_victoires': 0,
        'nuls': 0,
        'temps_mcts': 0,
        'temps_ab': 0
    }
    
    for i in range(parties):
        etat = jeu.etat_initial()
        # Alterner qui commence
        mcts_joueur = 'MAX' if i % 2 == 0 else 'MIN'
        ab_joueur = 'MIN' if mcts_joueur == 'MAX' else 'MAX'
        
        while not jeu.est_terminal(etat):
            if jeu.joueur(etat) == mcts_joueur:
                start = time.time()
                mcts = MCTS(jeu)
                action, _ = mcts.recherche(etat, iterations=mcts_iterations)
                stats['temps_mcts'] += time.time() - start
            else:
                start = time.time()
                _, action = alphabeta(jeu, etat, alpha_beta_depth, joueur_max=ab_joueur)
                stats['temps_ab'] += time.time() - start
            
            etat = jeu.resultat(etat, action)
        
        resultat = jeu.utilite(etat, mcts_joueur)
        if resultat == 1:
            stats['mcts_victoires'] += 1
        elif resultat == -1:
            stats['ab_victoires'] += 1
        else:
            stats['nuls'] += 1
    
    return stats


# Attention : Alpha-Beta profondeur 6 sur Connect Four peut etre lent
# On utilise profondeur 4 pour garder un temps raisonnable
cf = ConnectFour()
stats_cf = tournoi_mcts_vs_alphabeta(cf, mcts_iterations=500, alpha_beta_depth=4, parties=10)

print("=== Tournoi MCTS vs Alpha-Beta sur Connect Four ===")
print(f"MCTS (500 iter)  : {stats_cf['mcts_victoires']} victoires")
print(f"Alpha-Beta (d=4) : {stats_cf['ab_victoires']} victoires")
print(f"Nuls             : {stats_cf['nuls']}")
print(f"Temps MCTS       : {stats_cf['temps_mcts']:.2f}s total")
print(f"Temps Alpha-Beta : {stats_cf['temps_ab']:.2f}s total")

---

In [ ]:
# Exercice 2 : Rollout intelligent
# Rollout guide par heuristiques : privilegier le centre et les coins

class MCTSSmartRollout(MCTS):
    """MCTS avec rollout intelligent."""
    
    def _simulation(self, noeud):
        """Simulation avec choix heuristique au lieu d'aleatoire."""
        self.stats['simulations'] += 1
        
        etat = noeud.etat
        joueur_max_original = self.jeu.joueur(noeud.parent.etat) if noeud.parent else 'MAX'
        
        while not self.jeu.est_terminal(etat):
            actions = list(self.jeu.actions(etat))
            scores = [heuristique_tictactoe(etat, a, self.jeu) for a in actions]
            
            # Choix semi-aleatoire : melanger score heuristique + bruit
            bruit = [random.random() * 0.3 for _ in actions]
            scores_combines = [s + b for s, b in zip(scores, bruit)]
            meilleur_idx = scores_combines.index(max(scores_combines))
            action = actions[meilleur_idx]
            etat = self.jeu.resultat(etat, action)
        
        return self.jeu.utilite(etat, joueur_max_original)


def heuristique_tictactoe(etat, action, jeu):
    """
    Heuristique pour guider le rollout dans TicTacToe.
    Retourne un score pour une action donnee dans un etat.
    Privilegie : victoire immediate > bloquer adversaire > centre > coins > bords
    """
    grille = etat[0]
    joueur_courant = etat[1]  # 'X' ou 'O'
    adversaire = 'O' if joueur_courant == 'X' else 'X'
    
    # Position de l'action dans la grille
    positions = {
        4: 3,   # Centre - meilleur
        0: 2, 2: 2, 6: 2, 8: 2,  # Coins
        1: 1, 3: 1, 5: 1, 7: 1,  # Bords
    }
    score = positions.get(action, 0)
    
    # Bonus : verifier si l'action gagne immediatement
    grille_test = list(grille)
    grille_test[action] = joueur_courant
    lignes = [[0,1,2],[3,4,5],[6,7,8],[0,3,6],[1,4,7],[2,5,8],[0,4,8],[2,4,6]]
    for l in lignes:
        if action in l and grille_test[l[0]] == grille_test[l[1]] == grille_test[l[2]] == joueur_courant:
            score += 10  # Victoire immediate
    
    # Bonus : bloquer l'adversaire
    grille_test[action] = adversaire
    for l in lignes:
        if action in l and grille_test[l[0]] == grille_test[l[1]] == grille_test[l[2]] == adversaire:
            score += 5  # Bloquer une victoire adverse
    
    return score


# Comparaison : MCTS standard vs MCTS smart rollout
jeu = TicTacToe()
n_parties = 30

def jouer_mcts_vs_mcts(jeu, mcts1_class, mcts1_kwargs, mcts2_class, mcts2_kwargs,
                        iterations=500, parties=20):
    """Fait jouer deux versions de MCTS l'une contre l'autre."""
    stats = {'mcts1_victoires': 0, 'mcts2_victoires': 0, 'nuls': 0}
    
    for i in range(parties):
        etat = jeu.etat_initial()
        # Alterner qui commence
        mcts1_joueur = 'MAX' if i % 2 == 0 else 'MIN'
        
        while not jeu.est_terminal(etat):
            if jeu.joueur(etat) == mcts1_joueur:
                mcts = mcts1_class(jeu, **mcts1_kwargs)
            else:
                mcts = mcts2_class(jeu, **mcts2_kwargs)
            action, _ = mcts.recherche(etat, iterations=iterations)
            etat = jeu.resultat(etat, action)
        
        resultat = jeu.utilite(etat, mcts1_joueur)
        if resultat == 1:
            stats['mcts1_victoires'] += 1
        elif resultat == -1:
            stats['mcts2_victoires'] += 1
        else:
            stats['nuls'] += 1
    
    return stats

stats = jouer_mcts_vs_mcts(
    jeu,
    MCTSSmartRollout, {},
    MCTS, {},
    iterations=500, parties=30
)

print("MCTS Smart Rollout vs MCTS Standard :")
print(f"  Smart Rollout : {stats['mcts1_victoires']} victoires")
print(f"  Standard      : {stats['mcts2_victoires']} victoires")
print(f"  Nuls          : {stats['nuls']}")
print(f"  Taux victoire Smart : {(stats['mcts1_victoires'] + 0.5 * stats['nuls']) / 30 * 100:.1f}%")

---

In [ ]:
# Exercice 1 : Parametre d'exploration c
# Test de differentes valeurs de c et mesure de l'impact sur les performances
# c = 0.5 : exploitation dominante
# c = 1.0 : equilibre
# c = 1.41 : valeur theorique (sqrt(2))
# c = 2.0 : exploration dominante

def jouer_partie_mcts_vs_aleatoire(jeu, mcts_joueur='MAX', c=1.41, iterations=1000):
    """
    Fait jouer MCTS (parametre c donne) contre un joueur aleatoire.
    Retourne 1 si MCTS gagne, 0 si nul, -1 si MCTS perd.
    """
    etat = jeu.etat_initial()
    
    while not jeu.est_terminal(etat):
        if jeu.joueur(etat) == mcts_joueur:
            mcts = MCTS(jeu, c=c)
            action, _ = mcts.recherche(etat, iterations=iterations)
        else:
            actions = list(jeu.actions(etat))
            action = random.choice(actions)
        etat = jeu.resultat(etat, action)
    
    return jeu.utilite(etat, mcts_joueur)


def benchmark_parametre_c(jeu, valeurs_c, iterations=1000, parties=10):
    """
    Compare les performances de MCTS avec differentes valeurs de c.
    Retourne un DataFrame avec les resultats.
    """
    resultats = []
    
    for c in valeurs_c:
        victoires = 0
        nuls = 0
        defaites = 0
        temps_total = 0
        
        for _ in range(parties):
            start = time.time()
            # MCTS joue en premier (MAX), aleatoire en second (MIN)
            resultat = jouer_partie_mcts_vs_aleatoire(jeu, mcts_joueur='MAX', c=c, iterations=iterations)
            temps_total += time.time() - start
            
            if resultat == 1:
                victoires += 1
            elif resultat == 0:
                nuls += 1
            else:
                defaites += 1
        
        resultats.append({
            'c': c,
            'Victoires': victoires,
            'Nuls': nuls,
            'Defaites': defaites,
            'Taux victoire (%)': (victoires + 0.5 * nuls) / parties * 100,
            'Temps moyen (s)': temps_total / parties
        })
    
    return pd.DataFrame(resultats)


# Benchmark
jeu = TicTacToe()
df_c = benchmark_parametre_c(jeu, valeurs_c=[0.5, 1.0, 1.41, 2.0], iterations=500, parties=20)
display(df_c)

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(df_c['c'].astype(str), df_c['Taux victoire (%)'], color='steelblue')
axes[0].set_xlabel('Parametre c')
axes[0].set_ylabel('Taux de victoire (%)')
axes[0].set_title('Impact de c sur les performances')

axes[1].bar(df_c['c'].astype(str), df_c['Temps moyen (s)'], color='coral')
axes[1].set_xlabel('Parametre c')
axes[1].set_ylabel('Temps moyen (s)')
axes[1].set_title('Impact de c sur le temps de calcul')

plt.tight_layout()
plt.show()

print("\nInterpretation :")
print("- c faible (0.5) privilegie l'exploitation -> peut manquer de bonnes actions")
print("- c = 1.41 (sqrt(2)) est la valeur theorique optimale")
print("- c eleve (2.0) explore trop -> dilue l'information")